prepara la base y borra y recrea las dos colecciones:

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from database.setup_db import setup_database
from pipeline.limpiar import limpiar_csv
import pipeline.carga as pc

DB = "panteon"
db = setup_database(db_name=DB, drop_existing=True)

=== Inicializando Base de Datos: 'panteon' en mongodb://localhost:27017 ===
[*] Creando colección 'species' con validación $jsonSchema...
[*] Creando colección 'sequences' con validación $jsonSchema...

=== Configurando Índices Estratégicos ===
[*] Índices en colección 'species'...
[*] Índices en colección 'sequences'...

✓ Configuración completada con éxito.

--- Índices actuales en 'species' ---
 - _id_: SON([('_id', 1)])
 - idx_species_key_unique: SON([('species_key', 1)])
 - idx_kingdom_class: SON([('Kingdom', 1), ('Class', 1)])
 - idx_species_source: SON([('Source', 1)])

--- Índices actuales en 'sequences' ---
 - _id_: SON([('_id', 1)])
 - idx_sequence_id_unique: SON([('sequence_id', 1)])
 - idx_species_superfamily: SON([('species_key', 1), ('te_superfamily', 1)])
 - idx_te_hierarchy: SON([('te_class', 1), ('te_order', 1), ('te_superfamily', 1)])
 - idx_metrics_length: SON([('metrics.length', 1)])
 - idx_metrics_gc: SON([('metrics.gc_content', 1)])


carga las especies:

In [2]:
tabla = limpiar_csv("../data/PanTEon_Database_metadata_v1.6.1.csv")
insertadas, rechazadas = pc.cargar_species(db, tabla)
print("species:", insertadas, "insertadas |", len(rechazadas), "rechazadas")
print(rechazadas[:5])

species: 2790 insertadas | 0 rechazadas
[]


Celda 3, verifica:

In [3]:
sp = db.species
print("CSV limpio:", len(tabla), "| en MongoDB:", sp.count_documents({}))

for c in ["Phylum", "Class", "Order", "Family"]:
    print(c, "null:", sp.count_documents({c: {"$type": "null"}}))

for campo in ["Kingdom", "Source", "source_2"]:
    r = sp.aggregate([{"$group": {"_id": f"${campo}", "n": {"$sum": 1}}}, {"$sort": {"n": -1}}])
    print(campo, {x["_id"]: x["n"] for x in r})

print("Índices:", list(sp.index_information()))

CSV limpio: 2790 | en MongoDB: 2790
Phylum null: 3
Class null: 49
Order null: 48
Family null: 76
Kingdom {'Fungi': 1559, 'Animalia': 780, 'Plantae': 451}
Source {'RM2+MCH': 2404, 'Dfam': 386}
source_2 {None: 2744, 'Repbase': 45, 'Dfam': 1}
Índices: ['_id_', 'idx_species_key_unique', 'idx_kingdom_class', 'idx_species_source']
